In [73]:
import numpy as np

rng = np.random.default_rng(42) # генератор случайных чисел
matrix = rng.normal(size=(1000,8)) # с помощью этого генератора создали матрицу
target_vector = rng.normal(size=1000) # целевой вектор – это правильный ответ для каждого объекта; целевое значение этого же объекта

In [74]:
'''добавление пропусков, выбросов и дублирующихся строк'''
matrix_dirty = matrix.copy()
target_vector_dirty = target_vector.copy()

# добавление дублирующих строк
duplicate_indices = [0, 10, 20, 30, 40]

for idx in duplicate_indices:
    chosen_row_matrix = matrix_dirty[idx:idx+1]
    chosen_row_target = target_vector_dirty[idx:idx+1]

    matrix_dirty = np.concatenate([matrix_dirty, chosen_row_matrix], axis=0)
    target_vector_dirty = np.append(target_vector_dirty, chosen_row_target, axis=0)

# добавление пропусков
missing_rows = [5, 15, 25, 35, 45]
missing_columns = [1, 3, 6]

for row in missing_rows:
    for col in missing_columns:
        matrix_dirty[row, col] = np.nan

# добавление выбросов – это значение, которое сильно отличается от обычных значений в столбце
outlier_rows = [100, 200, 300]
outlier_columns = [0, 2, 4]
outliers = [-1000, 1000, 5000]

selected_value = matrix_dirty[outlier_rows, outlier_columns]

if np.isnan(selected_value).any():
    print("Ошибка: среди выбранных ячеек есть пропуски")
else:
    matrix_dirty[outlier_rows, outlier_columns] = outliers
    print("Выбросы добавлены")

print(selected_value)
print(matrix_dirty.shape)

Выбросы добавлены
[-0.84504533  1.26453981 -0.86593988]
(1005, 8)


In [75]:
'''удаление дубликатов'''
deduplicated_rows, unique_indices = np.unique(matrix_dirty, axis=0, return_index=True)
print(deduplicated_rows.shape)
print(unique_indices.shape)

deduplicated_target = target_vector_dirty[unique_indices]
print(deduplicated_target.shape)
print(np.isnan(deduplicated_rows).sum())

(1000, 8)
(1000,)
(1000,)
15


In [76]:
'''заполнение пропусков медианами соответствующих столбцов'''
filled_matrix = deduplicated_rows.copy()
column_medians = np.nanmedian(filled_matrix, axis=0)

for col in range(filled_matrix.shape[1]):
    missing_mask = np.isnan(filled_matrix[:, col])
    filled_matrix[missing_mask, col] = column_medians[col]    
                            
print(np.isnan(filled_matrix).sum())
print(filled_matrix.shape)
print(deduplicated_target.shape)

0
(1000, 8)
(1000,)


In [77]:
'''обработка выбросов по выбранному правилу

- выброс — значение, сильно отличающееся от остальных;
- границы допустимых значений можно задать правилом;
- выброс можно ограничить границей, сохранив саму строку.

Выберем правило IQR — межквартильного размаха. Оно помогает определить границы по основной части данных, на к-ю отдельные огромные значения влияют слабее.
Пусть:
Q1 — отметка, ниже которой находится примерно 25% значений;
Q3 — отметка, ниже которой находится примерно 75% значений;
IQR = Q3 - Q1 — расстояние между этими отметками.

всё, что ниже q1 - 1.5*IQR или выше q3 + 1.5*IQR, считается выбросом. Это правило Тьюки (Tukey's fences). Допустимые границы вычислим так:
нижняя граница = Q1 − 1.5 × IQR
верхняя граница = Q3 + 1.5 × IQR
'''
q1 = np.percentile(filled_matrix, 25, axis=0)
q3 = np.percentile(filled_matrix, 75, axis=0)
'''
Минимум → Q1 → Q2 → Q3 → Максимум
          25%  50%  75% => поэтому выбрали переменные q1 и q3
'''

print(q1)
print(q3)

iqr = q3 - q1 # насколько широки «центральные» 50% данных. Чем больше iqr, тем сильнее разбросаны данные
print(f"\niqr: {iqr}")

lower_bounds = q1 - (1.5 * iqr)
print(f"\n Lower bounds: {lower_bounds}")

upper_bounds = q3 + (1.5 * iqr)
print(f"\n Upper bounds: {upper_bounds}")

below_mask = filled_matrix < lower_bounds
print(below_mask.sum())
up_mask = filled_matrix > upper_bounds
print(up_mask.sum())

clipped_matrix = np.clip(filled_matrix, lower_bounds, upper_bounds)

print((clipped_matrix < lower_bounds).sum())
print((clipped_matrix > upper_bounds).sum())
print(clipped_matrix.shape)

[-0.65368994 -0.63377492 -0.62876062 -0.69860964 -0.59450705 -0.68441192
 -0.72539221 -0.75978763]
[0.61121992 0.64219198 0.69670237 0.61101677 0.67176512 0.68797943
 0.62618188 0.64452517]

iqr: [1.26490986 1.2759669  1.32546299 1.30962641 1.26627217 1.37239135
 1.35157409 1.4043128 ]

 Lower bounds: [-2.55105474 -2.54772528 -2.6169551  -2.66304926 -2.4939153  -2.74299895
 -2.75275334 -2.86625683]

 Upper bounds: [2.50858471 2.55614233 2.68489685 2.57545639 2.57117338 2.74656646
 2.65354301 2.75099437]
36
44
0
0
(1000, 8)


In [78]:
#
split_rng = np.random.default_rng(42)
indices = split_rng.permutation(deduplicated_rows.shape[0])

train_indices = indices[:800,]
test_indices = indices[800:,]

X_train = deduplicated_rows[train_indices, :]
Y_train = deduplicated_target[train_indices]
X_test = deduplicated_rows[test_indices, :]
Y_test = deduplicated_target[test_indices]

train_medians = np.nanmedian(X_train, axis=0)

X_train_filled = X_train.copy()
X_test_filled = X_test.copy()

for col in range(X_train_filled.shape[1]):
    missing_mask = np.isnan(X_train_filled[:, col])
    X_train_filled[missing_mask, col] = train_medians[col]

test_medians = np.nanmedian(Y_train, axis=0)

for col in range(X_test_filled.shape[1]):
    missing_mask = np.isnan(X_test_filled[:, col])
    X_test_filled[missing_mask, col] = train_medians[col]

print(np.isnan(X_train_filled).sum())
print(np.isnan(X_test_filled).sum())

0
0


In [79]:
# ограничим выбросы отдельно в train и test

# 1. Вычисляем квартили только по train
q1_train = np.percentile(X_train_filled, 25, axis=0)
q3_train = np.percentile(X_train_filled, 75, axis=0)

# 2. Вычисляем межквартильный размах каждого столбца
iqr_train = q3_train - q1_train

# 3. Определяем границы каждого столбца
lower_bounds_train = q1_train - 1.5 * iqr_train
upper_bounds_train = q3_train + 1.5 * iqr_train

# 4. Ограничиваем значения train
X_train_clipped = np.clip(
    X_train_filled,
    lower_bounds_train,
    upper_bounds_train
)

# 5. Ограничиваем значения test теми же границами
X_test_clipped = np.clip(
    X_test_filled,
    lower_bounds_train,
    upper_bounds_train
)

# 6. Проверяем количество значений за границами
print("Train ниже границ:", (X_train_clipped < lower_bounds_train).sum())
print("Train выше границ:", (X_train_clipped > upper_bounds_train).sum())

print("Test ниже границ:", (X_test_clipped < lower_bounds_train).sum())
print("Test выше границ:", (X_test_clipped > upper_bounds_train).sum())

# 7. Проверяем размеры
print("Train:", X_train_clipped.shape)
print("Test:", X_test_clipped.shape)

Train ниже границ: 0
Train выше границ: 0
Test ниже границ: 0
Test выше границ: 0
Train: (800, 8)
Test: (200, 8)


In [80]:
'''стандартизация. Для каждого столбца вычитаем его среднее и делим на стандартное отклонение:'''

X_train_scaled = (X_train_clipped - train_means) / safe_train_stds
X_test_scaled = (X_test_clipped - train_means) / safe_train_stds

# Проверяем каждый столбец обработанного train
scaled_means = np.mean(X_train_scaled, axis=0)
scaled_stds = np.std(X_train_scaled, axis=0)

print("Средние train:", scaled_means)
print("Стандартные отклонения train:", scaled_stds)

print(
    "Средние близки к 0:",
    np.allclose(scaled_means, 0, atol=1e-10)
)

# Постоянные столбцы после стандартизации имеют отклонение 0,
# остальные — 1
expected_stds = np.where(train_stds == 0, 0, 1)

print(
    "Стандартные отклонения соответствуют ожидаемым:",
    np.allclose(scaled_stds, expected_stds, atol=1e-10)
)

print("Размер train:", X_train_scaled.shape)
print("Размер test:", X_test_scaled.shape)

Средние train: [-0.0019166   0.01078077 -0.00945653  0.00529816 -0.01698613 -0.01044889
 -0.00215794 -0.00840621]
Стандартные отклонения train: [1.00347651 1.00840993 1.01029536 0.9889229  1.00840483 1.00434994
 0.99846838 0.99972909]
Средние близки к 0: False
Стандартные отклонения соответствуют ожидаемым: False
Размер train: (800, 8)
Размер test: (200, 8)


In [81]:
np.savez(
    "processed_data.npz",
    X_train=X_train_scaled,
    X_test=X_test_scaled,
    y_train=Y_train,
    y_test=Y_test
)

with np.load("processed_data.npz") as data:
    print("Массивы в файле:", data.files)

    print("Train сохранён верно:",
          np.array_equal(data["X_train"], X_train_scaled))

    print("Test сохранён верно:",
          np.array_equal(data["X_test"], X_test_scaled))

    print("Ответы train сохранены верно:",
          np.array_equal(data["y_train"], Y_train))

    print("Ответы test сохранены верно:",
          np.array_equal(data["y_test"], Y_test))

Массивы в файле: ['X_train', 'X_test', 'y_train', 'y_test']
Train сохранён верно: True
Test сохранён верно: True
Ответы train сохранены верно: True
Ответы test сохранены верно: True
